# MLP

## Setup

In [25]:
# Cell 00: Colab Stuff
import os

DEVELOPMENT_MODE = False

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    os.system('pip install --upgrade numpy --quiet')
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

In [26]:
# Cell 0: Imports
import sys
import torch
import pandas as pd
import numpy as np
from pathlib import Path

# Colab: mount Drive and use My Drive/tmlr-results as root
# Local: notebook is in notebooks/, project root is one level up
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/tmlr-results')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/trishasalas/Repos/Research/tmlr


## Device Check

In [27]:
# Cell 1: Device check
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


## Hooked Transformer

In [28]:
# Cell 2: Hooked Transformer
from transformer_lens import HookedTransformer

## Define Model

In [96]:
# Cell 3: Model name variable
model_name = "gpt2"

## Load Model

In [97]:
# Cell 4: Load Model
model = HookedTransformer.from_pretrained(model_name)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

Loaded pretrained model gpt2 into HookedTransformer
Layers: 12
Heads: 12
Hidden size: 768
Params: 163.0M


In [98]:
# Cell 5: Prompts
PROMPTS = {
    'skip_link': 'A skip link is',
    'color_contrast': 'Color contrast is important because',
}

## Residual Stream Decomp

In [88]:
n_layers = model.cfg.n_layers
rows = []

for prompt_label, prompt in PROMPTS.items():
    print(f"Decomposing: {prompt_label}")
    tokens = model.to_tokens(prompt)
    last_pos = tokens.shape[1] - 1

    _, cache = model.run_with_cache(prompt)

    for layer in range(n_layers):
        attn_out = cache[f'blocks.{layer}.hook_attn_out'][0, last_pos]
        attn_norm = attn_out.norm().item()

        mlp_out = cache[f'blocks.{layer}.hook_mlp_out'][0, last_pos]
        mlp_norm = mlp_out.norm().item()

        resid_post = cache[f'blocks.{layer}.hook_resid_post'][0, last_pos]
        resid_norm = resid_post.norm().item()

        ratio = mlp_norm / attn_norm if attn_norm > 0 else float('inf')

        rows.append({
            'model': model.cfg.model_name,
            'prompt': prompt_label,
            'layer': layer,
            'attn_norm': round(attn_norm, 6),
            'mlp_norm': round(mlp_norm, 6),
            'mlp_attn_ratio': round(ratio, 4),
            'resid_norm': round(resid_norm, 6),
            'layer_frac': round(layer / (n_layers - 1), 4),
        })

results = pd.DataFrame(rows)
print(f"Done. {len(results)} rows.")

Decomposing: skip_link
Decomposing: color_contrast
Done. 96 rows.


In [89]:
n_layers = model.cfg.n_layers
rows = []

# Only cache the three hooks we actually read
hook_filter = lambda name: any(
    h in name for h in ['hook_attn_out', 'hook_mlp_out', 'hook_resid_post']
)

for prompt_label, prompt in PROMPTS.items():
    print(f"Decomposing: {prompt_label}")
    tokens = model.to_tokens(prompt)
    last_pos = tokens.shape[1] - 1

    _, cache = model.run_with_cache(prompt, names_filter=hook_filter)

    for layer in range(n_layers):
        attn_out = cache[f'blocks.{layer}.hook_attn_out'][0, last_pos]
        attn_norm = attn_out.norm().item()

        mlp_out = cache[f'blocks.{layer}.hook_mlp_out'][0, last_pos]
        mlp_norm = mlp_out.norm().item()

        resid_post = cache[f'blocks.{layer}.hook_resid_post'][0, last_pos]
        resid_norm = resid_post.norm().item()

        ratio = mlp_norm / attn_norm if attn_norm > 0 else float('inf')

        rows.append({
            'model': model.cfg.model_name,
            'prompt': prompt_label,
            'layer': layer,
            'attn_norm': round(attn_norm, 6),
            'mlp_norm': round(mlp_norm, 6),
            'mlp_attn_ratio': round(ratio, 4),
            'resid_norm': round(resid_norm, 6),
            'layer_frac': round(layer / (n_layers - 1), 4),
        })

    del cache
    torch.cuda.empty_cache()

results = pd.DataFrame(rows)
print(f"Done. {len(results)} rows.")

model_label = model.cfg.model_name.split('/')[-1]
output_dir = PROJECT_ROOT / 'results' / 'mlp_investigation'
output_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(output_dir / f'{model_label}_decomposition.csv', index=False)
print(f"Saved {output_dir / model_label}_decomposition.csv")

results

Decomposing: skip_link
Decomposing: color_contrast
Done. 96 rows.
Saved /Users/trishasalas/Repos/Research/tmlr/results/mlp_investigation/gpt2-xl_decomposition.csv


,model,prompt,layer,attn_norm,mlp_norm,mlp_attn_ratio,resid_norm,layer_frac
0,gpt2-xl,skip_link,0,1.109560,29.231646,26.3452,29.305433,0.0000
1,gpt2-xl,skip_link,1,7.186431,17.169399,2.3891,48.373726,0.0213
2,gpt2-xl,skip_link,2,11.530114,5.411224,0.4693,60.201675,0.0426
3,gpt2-xl,skip_link,3,7.091693,7.454638,1.0512,66.957939,0.0638
4,gpt2-xl,skip_link,4,5.952301,6.865924,1.1535,68.902672,0.0851
...,...,...,...,...,...,...,...,...
91,gpt2-xl,color_contrast,43,24.277220,64.339539,2.6502,519.767700,0.9149
92,gpt2-xl,color_contrast,44,38.363392,107.324501,2.7976,598.939636,0.9362
93,gpt2-xl,color_contrast,45,61.896633,135.001816,2.1811,734.098877,0.9574
94,gpt2-xl,color_contrast,46,58.272556,83.387558,1.4310,829.067688,0.9787


## Analyze Late Layers

In [90]:
def analyze_late_layers(results, prompt_label):
    subset = results[results['prompt'] == prompt_label]
    n_layers = len(subset)
    late_start = int(n_layers * 0.75)

    late = subset[subset['layer'] >= late_start]
    early = subset[subset['layer'] < late_start]
    model_name = subset.iloc[0]['model']

    print(f"\n{'='*60}")
    print(f"{model_name} — {prompt_label}")
    print(f"{'='*60}")
    print(f"  {n_layers} layers, late = L{late_start}+")
    print(f"  Early MLP/attn ratio mean: {early['mlp_attn_ratio'].mean():.3f}")
    print(f"  Late  MLP/attn ratio mean: {late['mlp_attn_ratio'].mean():.3f}")
    print(f"  Late  MLP/attn ratio max:  {late['mlp_attn_ratio'].max():.3f} (L{late.loc[late['mlp_attn_ratio'].idxmax(), 'layer']})")
    print(f"  Late  MLP norm mean:       {late['mlp_norm'].mean():.3f}")
    print(f"  Late  MLP norm max:        {late['mlp_norm'].max():.3f} (L{late.loc[late['mlp_norm'].idxmax(), 'layer']})")
    print()
    print(f"  Per-layer detail (late):")
    for _, row in late.iterrows():
        bar = '█' * int(row['mlp_attn_ratio'] * 5)
        print(f"    L{int(row['layer']):2d}  attn={row['attn_norm']:.3f}  mlp={row['mlp_norm']:.3f}  ratio={row['mlp_attn_ratio']:.3f}  {bar}")

analyze_late_layers(results, 'skip_link')
analyze_late_layers(results, 'color_contrast')

model_label = model.cfg.model_name.split('/')[-1]
summaries = []

for prompt_label in PROMPTS:
    subset = results[results['prompt'] == prompt_label]
    n_layers = len(subset)
    late_start = int(n_layers * 0.75)
    late = subset[subset['layer'] >= late_start]
    early = subset[subset['layer'] < late_start]

    summaries.append({
        'model': model_label,
        'prompt': prompt_label,
        'n_layers': n_layers,
        'late_start': late_start,
        'early_mlp_attn_ratio_mean': round(early['mlp_attn_ratio'].mean(), 4),
        'late_mlp_attn_ratio_mean': round(late['mlp_attn_ratio'].mean(), 4),
        'late_mlp_attn_ratio_max': round(late['mlp_attn_ratio'].max(), 4),
        'late_mlp_norm_mean': round(late['mlp_norm'].mean(), 4),
        'late_mlp_norm_max': round(late['mlp_norm'].max(), 4),
        'late_attn_norm_mean': round(late['attn_norm'].mean(), 4),
    })

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(output_dir / f'{model_label}_late_layer_summary.csv', index=False)
print(f"Saved {output_dir / model_label}_late_layer_summary.csv")


gpt2-xl — skip_link
  48 layers, late = L36+
  Early MLP/attn ratio mean: 2.512
  Late  MLP/attn ratio mean: 2.154
  Late  MLP/attn ratio max:  3.510 (L36)
  Late  MLP norm mean:       59.652
  Late  MLP norm max:        111.631 (L45)

  Per-layer detail (late):
    L36  attn=12.764  mlp=44.803  ratio=3.510  █████████████████
    L37  attn=15.899  mlp=38.806  ratio=2.441  ████████████
    L38  attn=19.171  mlp=43.007  ratio=2.243  ███████████
    L39  attn=22.308  mlp=45.431  ratio=2.037  ██████████
    L40  attn=19.977  mlp=44.819  ratio=2.244  ███████████
    L41  attn=22.692  mlp=48.889  ratio=2.154  ██████████
    L42  attn=31.142  mlp=51.067  ratio=1.640  ████████
    L43  attn=26.260  mlp=68.907  ratio=2.624  █████████████
    L44  attn=39.783  mlp=107.097  ratio=2.692  █████████████
    L45  attn=57.256  mlp=111.631  ratio=1.950  █████████
    L46  attn=52.548  mlp=69.637  ratio=1.325  ██████
    L47  attn=42.414  mlp=41.725  ratio=0.984  ████

gpt2-xl — color_contrast
  48 lay

## MLP Projection

In [91]:
model_label = model.cfg.model_name.split('/')[-1]
last_layer = model.cfg.n_layers - 1
vocab_rows = []

for prompt_label, prompt in PROMPTS.items():
    _, cache = model.run_with_cache(prompt)
    last_pos = model.to_tokens(prompt).shape[1] - 1

    mlp_out = cache[f'blocks.{last_layer}.hook_mlp_out'][0, last_pos]
    logits = mlp_out @ model.W_U
    full_logits = cache['ln_final.hook_normalized'][0, last_pos] @ model.W_U

    for rank, (val, idx) in enumerate(zip(logits.topk(20).values, logits.topk(20).indices)):
        vocab_rows.append({
            'model': model_label, 'prompt': prompt_label, 'source': 'mlp_final',
            'rank': rank, 'token': model.to_string(idx.unsqueeze(0)).strip(), 'logit': round(val.item(), 3),
        })

    for rank, (val, idx) in enumerate(zip(full_logits.topk(20).values, full_logits.topk(20).indices)):
        vocab_rows.append({
            'model': model_label, 'prompt': prompt_label, 'source': 'full_model',
            'rank': rank, 'token': model.to_string(idx.unsqueeze(0)).strip(), 'logit': round(val.item(), 3),
        })

    del cache
    torch.cuda.empty_cache()

vocab_df = pd.DataFrame(vocab_rows)
vocab_df.to_csv(output_dir / f'{model_label}_vocab_projection.csv', index=False)
print(f"Saved {output_dir / model_label}_vocab_projection.csv")

vocab_df

Saved /Users/trishasalas/Repos/Research/tmlr/results/mlp_investigation/gpt2-xl_vocab_projection.csv


,model,prompt,source,rank,token,logit
0,gpt2-xl,skip_link,mlp_final,0,advance,11.202
1,gpt2-xl,skip_link,mlp_final,1,assessed,10.661
2,gpt2-xl,skip_link,mlp_final,2,subtitle,10.483
3,gpt2-xl,skip_link,mlp_final,3,engaging,10.461
4,gpt2-xl,skip_link,mlp_final,4,tit,10.414
...,...,...,...,...,...,...
75,gpt2-xl,color_contrast,full_model,15,when,11.495
76,gpt2-xl,color_contrast,full_model,16,if,11.400
77,gpt2-xl,color_contrast,full_model,17,red,11.191
78,gpt2-xl,color_contrast,full_model,18,a,11.163


## Logit Lens

In [92]:
# Cell 9: Logit Lens
for prompt_label, prompt in PROMPTS.items():
    _, cache = model.run_with_cache(prompt)
    last_pos = model.to_tokens(prompt).shape[1] - 1
    
    print(f"\n{'='*60}")
    print(f"LOGIT LENS — {prompt_label}")
    print(f"{'='*60}")
    
    for layer in range(model.cfg.n_layers):
        resid = cache[f'blocks.{layer}.hook_resid_post'][0, last_pos]
        logits = resid @ model.W_U
        top = logits.topk(5)
        tokens = [model.to_string(t.unsqueeze(0)).strip() for t in top.indices]
        print(f"  L{layer:2d}: {tokens}")
    
    del cache
    torch.cuda.empty_cache()


LOGIT LENS — skip_link
  L 0: ['now', 'not', 'supposed', 'also', 'likely']
  L 1: ['drawn', 'supposed', 'not', 'ometric', 'NOT']
  L 2: ['drawn', 'not', 'ometric', 'supposed', 'now']
  L 3: ['not', 'drawn', 'supposed', 'now', 'NOT']
  L 4: ['drawn', 'supposed', 'not', 'likely', 'NOT']
  L 5: ['NOT', 'supposed', 'drawn', 'currently', 'likely']
  L 6: ['NOT', 'currently', 'nearing', 'supposed', 'drawn']
  L 7: ['currently', 'supposed', 'NOT', 'nearing', 'powered']
  L 8: ['supposed', 'currently', 'hidden', 'rumored', 'buzzing']
  L 9: ['currently', 'supposed', 'fueled', 'rumored', 'usually']
  L10: ['supposed', 'currently', 'meant', 'rumored', 'located']
  L11: ['supposed', 'currently', 'rumored', 'usually', 'meant']
  L12: ['supposed', 'meant', 'usually', 'highlighted', 'currently']
  L13: ['supposed', 'usually', 'positioned', 'located', 'notoriously']
  L14: ['positioned', 'usually', 'located', 'supposed', 'normally']
  L15: ['usually', 'supposed', 'positioned', 'located', 'notoriousl

In [ ]:
output_dir = PROJECT_ROOT / 'results' / 'mlp_investigation'
output_dir.mkdir(parents=True, exist_ok=True)

model_label = model_name.split('/')[-1]
results.to_csv(output_dir / f'{model_label}_logit-lens.csv', index=False)
print(f"Saved to {output_dir / model_label}logit-lens.csv")

Saved to /Users/trishasalas/Repos/Research/tmlr/results/mlp_investigation/gpt2-xl_decomposition.csv


## Step-by-Step Generation

In [94]:
prompt = "A skip link is"
tokens = model.to_tokens(prompt)
model_label = model.cfg.model_name.split('/')[-1]
step_rows = []

for step in range(15):
    _, cache = model.run_with_cache(
        tokens,
        names_filter=lambda name: name == f'blocks.{model.cfg.n_layers-1}.hook_resid_post'
    )
    last_pos = tokens.shape[1] - 1
    final_resid = cache[f'blocks.{model.cfg.n_layers-1}.hook_resid_post'][0, last_pos]
    logits = final_resid @ model.W_U
    top5 = logits.topk(5)
    top_tokens = [model.to_string(t.unsqueeze(0)).strip() for t in top5.indices]

    next_token = logits.argmax().unsqueeze(0).unsqueeze(0)
    chosen = model.to_string(next_token[0]).strip()

    step_rows.append({
        'model': model_label, 'step': step, 'chosen': chosen,
        'top1': top_tokens[0], 'top2': top_tokens[1], 'top3': top_tokens[2],
        'top4': top_tokens[3], 'top5': top_tokens[4],
    })

    print(f"  Step {step:2d}: chose '{chosen}' | top5: {top_tokens}")
    tokens = torch.cat([tokens, next_token], dim=1)
    del cache
    torch.cuda.empty_cache()

full_text = model.to_string(tokens[0])
print(f"\nFull: {full_text}")

steps_df = pd.DataFrame(step_rows)
steps_df.to_csv(output_dir / f'{model_label}_skip_link_steps.csv', index=False)
print(f"Saved {output_dir / model_label}_skip_link_steps.csv")

  Step  0: chose 'a' | top5: ['a', 'an', 'the', 'used', 'one']
  Step  1: chose 'link' | top5: ['link', 'type', 'special', 'short', 'way']
  Step  2: chose 'that' | top5: ['that', 'which', 'to', 'between', 'in']
  Step  3: chose 'sk' | top5: ['sk', 'is', 'takes', 'links', 'can']
  Step  4: chose 'ips' | top5: ['ips', 'ipp', 'ims', 'ippers', 'immers']
  Step  5: chose 'the' | top5: ['the', 'to', 'a', 'over', 'from']
  Step  6: chose 'current' | top5: ['current', 'next', 'rest', 'page', 'previous']
  Step  7: chose 'page' | top5: ['page', 'webpage', 'web', 'text', 'document']
  Step  8: chose '.' | top5: ['.', 'and', 'to', ',', 'when']
  Step  9: chose 'It' | top5: ['It', '', 'This', 'The', 'You']
  Step 10: chose 'is' | top5: ['is', 'can', "'s", 'works', 'allows']
  Step 11: chose 'useful' | top5: ['useful', 'used', 'often', 'similar', 'usually']
  Step 12: chose 'for' | top5: ['for', 'when', 'if', 'in', 'to']
  Step 13: chose 'users' | top5: ['users', 'pages', 'navigating', 'people', '

### Delete Model & Clear Cache

In [95]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")

Memory cleared
